# Figure 6 — public-data treatment-interaction analysis

This notebook reconstructs Figure 6 directly from the public JAVELIN Renal
101 and CheckMate 025 supplementary workbooks downloaded by
`00_Paper_Data_Download.ipynb`. It does not import archived project results and
does not compare any estimate, confidence interval, P value, or FDR with a
previously reported value.

Analysis plan:

- Gene scores are frozen before outcome modeling. Each component gene is
  standardized within cohort; multi-gene scores are the equal-weight mean of
  their component z scores and are then standardized to one cohort SD.
- JAVELIN PFS models use Cox proportional hazards regression. Panel A contains
  descriptive median-split subgroups. Panel B tests prespecified continuous
  score-by-treatment interactions adjusted for age, sex, and PD-L1. Its BH FDR
  family contains nine molecular scores, although only four are plotted.
- Panel C fits exploratory Mitoxy-by-treatment interactions within median
  immune/angiogenic strata, with clinical and molecular-context adjustment.
- CheckMate 025 uses logistic regression for ORR and Cox models for PFS/OS.
  Panel D reports Mitoxy from a prespecified family of eight scores × three
  outcomes; BH correction is performed across all 24 interaction tests.

The compact output CSV names and columns are unchanged, so the existing
Figure 6 plotting notebook can be rerun without modification.


In [19]:
from pathlib import Path
import hashlib
import os
import re
import warnings

import numpy as np
import pandas as pd
import statsmodels.api as sm
from openpyxl import load_workbook
from pyxlsb import open_workbook as open_xlsb
from statsmodels.duration.hazard_regression import PHReg
from scipy.stats import norm


PROJECT_ROOT = Path(
    os.environ.get("MITOXY_PROJECT_ROOT", "/scratch/ar58064/mitoxy")
).resolve()
MANIFEST_PATH = (
    PROJECT_ROOT / "data" / "provenance" /
    "paper_public_source_manifest.csv"
)
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "figure_6_clean"
CACHE_DIR = PROJECT_ROOT / "data" / "processed" / "figure_6_public_cache"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

manifest = pd.read_csv(MANIFEST_PATH)


def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_size), b""):
            digest.update(chunk)
    return digest.hexdigest()


def source_path(dataset, role):
    match = manifest.loc[
        manifest["dataset"].eq(dataset)
        & manifest["role"].eq(role)
    ]
    if len(match) != 1:
        raise RuntimeError(
            f"Expected one source for ({dataset!r}, {role!r}); "
            f"found {len(match)}"
        )
    row = match.iloc[0]
    path = PROJECT_ROOT / str(row["relative_path"])
    if not path.is_file():
        raise FileNotFoundError(path)
    if path.stat().st_size != int(row["size_bytes"]):
        raise RuntimeError(f"Source-size mismatch: {path}")
    if sha256_file(path) != str(row["sha256"]):
        raise RuntimeError(f"Source SHA-256 mismatch: {path}")
    return path


RAW = {
    "javelin": source_path(
        "JAVELIN Renal 101", "clinical and molecular tables"
    ),
    "checkmate": source_path(
        "CheckMate 025", "clinical and molecular tables"
    ),
}

display(pd.DataFrame({"analysis_key": RAW.keys(), "path": RAW.values()}))
print("Public-source integrity audit: PASS")


,analysis_key,path
0,javelin,/scratch/ar58064/mitoxy/data/raw/javelin_renal...
1,checkmate,/scratch/ar58064/mitoxy/data/raw/checkmate_025...


Public-source integrity audit: PASS


In [20]:
MITOXY_GENES = ["BAK1", "BAX", "BID", "SLC7A11"]
CORE3_GENES = ["BAK1", "BAX", "BID"]
MYELOID_GENES = ["MYD88", "TNF", "TLR2", "TLR4"]
SPATIAL4_GENES = ["MAPKAP1", "MLST8", "MTOR", "RICTOR"]
ANGIO_GENES = ["VEGFA", "KDR", "ESM1", "PECAM1", "ANGPTL4", "CD34"]
JAVELIN26_GENES = [
    "CD3G", "CD3E", "CD8B", "THEMIS", "TRAT1", "GRAP2", "CD247",
    "CD2", "CD96", "PRF1", "CD6", "IL7R", "ITK", "GPR18", "EOMES",
    "SIT1", "NLRC3", "CD244", "KLRD1", "SH2D1A", "CCL5", "XCL2",
    "CST7", "GFI1", "KCNA3", "PSTPIP1",
]

JAVELIN_TARGET_GENES = list(dict.fromkeys(
    MITOXY_GENES + MYELOID_GENES + SPATIAL4_GENES +
    ANGIO_GENES + JAVELIN26_GENES
))
CHECKMATE_TARGET_GENES = list(dict.fromkeys(
    MITOXY_GENES + MYELOID_GENES + SPATIAL4_GENES
))


def normalized(text):
    return re.sub(r"[^a-z0-9]+", " ", str(text).lower()).strip()


def canonical_id(value):
    if pd.isna(value):
        return ""
    text = re.sub(r"[^A-Za-z0-9]", "", str(value)).upper()
    if len(text) == 33 and text.startswith("X"):
        text = text[1:]
    return text


def unique_headers(values):
    output, counts = [], {}
    for index, value in enumerate(values):
        base = str(value).strip() if value not in (None, "") else f"column_{index}"
        counts[base] = counts.get(base, 0) + 1
        output.append(base if counts[base] == 1 else f"{base}_{counts[base]}")
    return output


def zscore_series(values):
    values = pd.to_numeric(values, errors="coerce").astype(float)
    standard_deviation = values.std(ddof=0)
    if not np.isfinite(standard_deviation) or standard_deviation == 0:
        raise ValueError("Cannot standardize a constant or invalid variable")
    return (values - values.mean()) / standard_deviation


def zscore_frame(frame):
    result = frame.apply(pd.to_numeric, errors="coerce").astype(float)
    standard_deviation = result.std(axis=0, ddof=0).replace(0, np.nan)
    result = result.subtract(result.mean(axis=0), axis=1).divide(
        standard_deviation, axis=1
    )
    if result.isna().any().any():
        bad = result.columns[result.isna().any()].tolist()
        raise ValueError(f"Cannot standardize genes with missing/invalid values: {bad}")
    return result


def bh_adjust(values):
    values = np.asarray(values, dtype=float)
    adjusted = np.full(values.shape, np.nan)
    valid = np.flatnonzero(np.isfinite(values))
    if len(valid) == 0:
        return adjusted
    order = valid[np.argsort(values[valid])]
    ranked = values[order] * len(order) / np.arange(1, len(order) + 1)
    ranked = np.minimum.accumulate(ranked[::-1])[::-1]
    adjusted[order] = np.minimum(ranked, 1.0)
    return adjusted


def resolve_numbered_sheet(sheet_names, number):
    patterns = [
        rf"(?:table|supplementary table|supp table|st|s)\s*0*{number}(?:\D|$)",
        rf"^0*{number}(?:\D|$)",
    ]
    matches = []
    for sheet in sheet_names:
        name = normalized(sheet)
        if any(re.search(pattern, name) for pattern in patterns):
            matches.append(sheet)
    if len(matches) == 1:
        return matches[0]
    contains = [sheet for sheet in sheet_names if str(number) in str(sheet)]
    if len(contains) == 1:
        return contains[0]
    raise RuntimeError(
        f"Could not uniquely resolve Supplementary Table {number}. "
        f"Candidates={matches or contains}; sheets={sheet_names}"
    )


def select_column(columns, alternatives, exclude=(), required=True):
    normalized_columns = {column: normalized(column) for column in columns}
    exclude = [normalized(term) for term in exclude]
    scored = []
    for column, text in normalized_columns.items():
        if any(term in text for term in exclude):
            continue
        best = 0
        for alternative in alternatives:
            terms = [normalized(term) for term in alternative]
            if all(term in text for term in terms):
                best = max(best, 100 + 10 * len(terms) - len(text))
        if best:
            scored.append((best, column))
    if not scored:
        if required:
            raise KeyError(
                f"No column matched {alternatives}. Available: {list(columns)}"
            )
        return None
    return max(scored, key=lambda item: (item[0], str(item[1])))[1]


def parse_event(values, column_name):
    numeric = pd.to_numeric(values, errors="coerce")
    observed = set(numeric.dropna().unique())
    name = normalized(column_name)
    if observed and observed.issubset({0, 1}):
        return 1 - numeric if ("censor" in name or "cnsr" in name) else numeric
    text = values.astype(str).str.strip().str.casefold()
    result = pd.Series(np.nan, index=values.index, dtype=float)
    result.loc[text.str.contains(
        r"death|dead|progress|event|yes|^1$", regex=True, na=False
    )] = 1.0
    result.loc[text.str.contains(
        r"censor|alive|no event|no progression|^0$", regex=True, na=False
    )] = 0.0
    return result


def parse_response(values):
    numeric = pd.to_numeric(values, errors="coerce")
    observed = set(numeric.dropna().unique())
    if observed and observed.issubset({0, 1}):
        return numeric.astype(float)
    text = values.astype(str).str.strip().str.upper()
    compact = text.str.replace(r"[^A-Z0-9]+", "", regex=True)
    result = pd.Series(np.nan, index=values.index, dtype=float)
    result.loc[
        compact.isin({"CR", "PR", "CRPR"})
        | text.str.contains(
            r"COMPLETE RESPONSE|PARTIAL RESPONSE|RESPONDER",
            regex=True, na=False,
        )
    ] = 1.0
    result.loc[
        compact.isin({"SD", "PD"})
        | text.str.contains(
            r"STABLE|PROGRESSIVE|NON.?RESPONDER",
            regex=True, na=False,
        )
    ] = 0.0
    return result


def prepare_design(data, score_column, categorical_covariates=(),
                   numeric_covariates=()):
    required = [
        score_column, "treatment", *categorical_covariates,
        *numeric_covariates,
    ]
    complete = data[required].replace([np.inf, -np.inf], np.nan).dropna().copy()
    design = pd.DataFrame(index=complete.index)
    design["score"] = pd.to_numeric(complete[score_column], errors="coerce")
    design["treatment"] = pd.to_numeric(complete["treatment"], errors="coerce")
    for column in numeric_covariates:
        design[column] = zscore_series(complete[column])
    if categorical_covariates:
        dummies = pd.get_dummies(
            complete[list(categorical_covariates)].astype(str),
            prefix=list(categorical_covariates), drop_first=True, dtype=float,
        )
        design = pd.concat([design, dummies], axis=1)
    design["score_by_treatment"] = design["score"] * design["treatment"]
    return complete, design.astype(float)


def cox_interaction(data, score_column, time_column, event_column,
                    categorical_covariates=(), numeric_covariates=()):
    required = [time_column, event_column]
    analysis = data.dropna(subset=required).copy()
    complete, design = prepare_design(
        analysis, score_column, categorical_covariates, numeric_covariates
    )
    outcomes = analysis.loc[complete.index, [time_column, event_column]].astype(float)
    valid = outcomes[time_column].gt(0) & outcomes[event_column].isin([0, 1])
    outcomes, design = outcomes.loc[valid], design.loc[valid]
    fit = PHReg(
        outcomes[time_column], design,
        status=outcomes[event_column], ties="breslow",
    ).fit(disp=0)
    index = design.columns.get_loc("score_by_treatment")
    coefficient = float(fit.params[index])
    standard_error = float(fit.bse[index])
    return {
        "n": len(outcomes),
        "events": int(outcomes[event_column].sum()),
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[index]),
    }


def cox_treatment_effect(data, time_column="pfs", event_column="pfs_event"):
    complete = data[[time_column, event_column, "treatment"]].dropna().copy()
    complete = complete.loc[
        complete[time_column].gt(0) & complete[event_column].isin([0, 1])
    ]
    design = complete[["treatment"]].astype(float)
    fit = PHReg(
        complete[time_column].astype(float), design,
        status=complete[event_column].astype(float), ties="breslow",
    ).fit(disp=0)
    coefficient, standard_error = float(fit.params[0]), float(fit.bse[0])
    return {
        "n": len(complete),
        "events": int(complete[event_column].sum()),
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[0]),
    }


def logistic_interaction(data, score_column, response_column="orr"):
    analysis = data.dropna(subset=[response_column]).copy()
    complete, design = prepare_design(analysis, score_column)
    outcome = analysis.loc[complete.index, response_column].astype(float)
    valid = outcome.isin([0, 1])
    outcome, design = outcome.loc[valid], design.loc[valid]
    design = sm.add_constant(design, has_constant="add")
    fit = sm.Logit(outcome, design).fit(disp=False, maxiter=500)
    coefficient = float(fit.params["score_by_treatment"])
    standard_error = float(fit.bse["score_by_treatment"])
    return {
        "n": len(outcome),
        "events": np.nan,
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues["score_by_treatment"]),
    }


def validate_intervals(table, estimate):
    if not table[estimate].gt(0).all():
        raise ValueError(f"Non-positive estimates in {estimate}")
    if not table["ci_low"].gt(0).all():
        raise ValueError("Non-positive confidence limit")
    if not table["ci_high"].gt(table["ci_low"]).all():
        raise ValueError("Invalid confidence-interval ordering")
    if not (
        table["ci_low"].le(table[estimate])
        & table[estimate].le(table["ci_high"])
    ).all():
        raise ValueError("Estimate falls outside its confidence interval")


## Reconstruct the JAVELIN public clinical and expression tables


In [21]:
JAVELIN_CLINICAL_CACHE = CACHE_DIR / "javelin_table11_clinical.csv"
JAVELIN_EXPRESSION_CACHE = CACHE_DIR / "javelin_table13_target_expression.csv"


def xlsb_values(cell_row):
    return [cell.v for cell in cell_row]


def reconstruct_javelin_clinical(path):
    excel = pd.ExcelFile(path, engine="pyxlsb")
    sheet = resolve_numbered_sheet(excel.sheet_names, 11)
    rows = []
    with open_xlsb(path) as workbook:
        with workbook.get_sheet(sheet) as worksheet:
            for row in worksheet.rows():
                rows.append(xlsb_values(row))
    header_scores = []
    for index, row in enumerate(rows[:40]):
        tokens = {normalized(value) for value in row if value not in (None, "")}
        score = sum(any(key in token for token in tokens) for key in [
            "id", "pfs", "age", "sex", "trt", "pdl1", "cluster",
        ])
        header_scores.append((score, index))
    score, header_index = max(header_scores)
    if score < 4:
        raise RuntimeError(
            f"Could not identify the JAVELIN clinical header in {sheet!r}"
        )
    width = max(len(row) for row in rows[header_index:])
    header = unique_headers(rows[header_index] + [None] * (width - len(rows[header_index])))
    body = [row + [None] * (width - len(row)) for row in rows[header_index + 1:]]
    table = pd.DataFrame(body, columns=header).dropna(how="all").dropna(axis=1, how="all")
    id_column = select_column(table.columns, [["id"]])
    table["canonical_id"] = table[id_column].map(canonical_id)
    table = table.loc[table["canonical_id"].str.len().gt(0)].drop_duplicates("canonical_id")
    print("JAVELIN clinical sheet:", sheet)
    print("JAVELIN clinical rows:", len(table))
    return table


def reconstruct_javelin_expression(path, clinical_ids, targets):
    excel = pd.ExcelFile(path, engine="pyxlsb")
    sheet = resolve_numbered_sheet(excel.sheet_names, 13)
    targets = set(targets)
    first_rows, target_rows = [], {}
    row_count = 0
    with open_xlsb(path) as workbook:
        with workbook.get_sheet(sheet) as worksheet:
            for row_index, row in enumerate(worksheet.rows()):
                values = xlsb_values(row)
                row_count += 1
                if row_index < 30:
                    first_rows.append(values)
                gene = next((
                    str(value).strip().upper()
                    for value in values[:12]
                    if str(value).strip().upper() in targets
                ), None)
                if gene is not None:
                    target_rows.setdefault(gene, []).append(values)
    header_candidates = []
    for index, row in enumerate(first_rows):
        overlap = sum(canonical_id(value) in clinical_ids for value in row)
        header_candidates.append((overlap, index, row))
    overlap, header_index, header = max(header_candidates, key=lambda item: item[0])
    if overlap < 100:
        raise RuntimeError(
            f"JAVELIN Table 13 identifier overlap is too small: {overlap}"
        )
    positions = [
        (index, canonical_id(value))
        for index, value in enumerate(header)
        if canonical_id(value) in clinical_ids
    ]
    expression = pd.DataFrame(index=[sample for _, sample in positions])
    for gene, rows in target_rows.items():
        vectors = []
        for row in rows:
            vectors.append([
                pd.to_numeric(row[index], errors="coerce")
                if index < len(row) else np.nan
                for index, _ in positions
            ])
        expression[gene] = np.nanmean(np.asarray(vectors, dtype=float), axis=0)
    expression.index.name = "canonical_id"
    expression = expression.reset_index().drop_duplicates("canonical_id")
    print("JAVELIN expression sheet:", sheet)
    print("Rows streamed:", row_count)
    print("Detected identifier row:", header_index)
    print("Identifier overlap:", overlap)
    print("Target genes recovered:", sorted(target_rows))
    return expression


if JAVELIN_CLINICAL_CACHE.exists():
    javelin_clinical = pd.read_csv(JAVELIN_CLINICAL_CACHE, dtype=str)
    print("Loading cached JAVELIN clinical table.")
else:
    javelin_clinical = reconstruct_javelin_clinical(RAW["javelin"])
    javelin_clinical.to_csv(JAVELIN_CLINICAL_CACHE, index=False)

clinical_ids = set(javelin_clinical["canonical_id"].dropna().map(canonical_id))
if JAVELIN_EXPRESSION_CACHE.exists():
    javelin_expression = pd.read_csv(JAVELIN_EXPRESSION_CACHE)
    cached_genes = set(javelin_expression.columns) - {"canonical_id"}
    cached_overlap = len(set(javelin_expression["canonical_id"].map(canonical_id)) & clinical_ids)
    if not set(JAVELIN_TARGET_GENES).issubset(cached_genes) or cached_overlap < 100:
        print("Discarding incomplete JAVELIN expression cache.")
        JAVELIN_EXPRESSION_CACHE.unlink()
        javelin_expression = reconstruct_javelin_expression(
            RAW["javelin"], clinical_ids, JAVELIN_TARGET_GENES
        )
        javelin_expression.to_csv(JAVELIN_EXPRESSION_CACHE, index=False)
    else:
        print("Loading cached JAVELIN target-expression table.")
else:
    javelin_expression = reconstruct_javelin_expression(
        RAW["javelin"], clinical_ids, JAVELIN_TARGET_GENES
    )
    javelin_expression.to_csv(JAVELIN_EXPRESSION_CACHE, index=False)

javelin_expression["canonical_id"] = javelin_expression["canonical_id"].map(canonical_id)
javelin_clinical["canonical_id"] = javelin_clinical["canonical_id"].map(canonical_id)
missing = sorted(set(JAVELIN_TARGET_GENES) - set(javelin_expression.columns))
if missing:
    raise KeyError(f"JAVELIN Table 13 is missing target genes: {missing}")

print("JAVELIN clinical identifiers:", len(clinical_ids))
print("JAVELIN expression identifiers:", javelin_expression["canonical_id"].nunique())
print("JAVELIN clinical/expression overlap:", len(
    set(javelin_expression["canonical_id"]) & clinical_ids
))


Loading cached JAVELIN clinical table.
Loading cached JAVELIN target-expression table.
JAVELIN clinical identifiers: 886
JAVELIN expression identifiers: 726
JAVELIN clinical/expression overlap: 726


In [22]:
# The public JAVELIN clinical table uses compact field names in the released
# workbook. Resolve exact aliases first, then semantic fallbacks.
def exact_or_semantic(columns, exact_aliases, alternatives, exclude=()):
    lookup = {normalized(column): column for column in columns}
    for alias in exact_aliases:
        if normalized(alias) in lookup:
            return lookup[normalized(alias)]
    return select_column(columns, alternatives, exclude=exclude)


j_id = exact_or_semantic(javelin_clinical.columns, ["ID"], [["id"]])
j_pfs = exact_or_semantic(
    javelin_clinical.columns, ["PFS_P"],
    [["pfs", "time"], ["pfs", "month"], ["pfs"]],
    exclude=["cnsr", "censor", "event", "status"],
)
j_pfs_status = exact_or_semantic(
    javelin_clinical.columns, ["PFS_P_CNSR"],
    [["pfs", "cnsr"], ["pfs", "censor"], ["pfs", "event"], ["pfs", "status"]],
)
j_age = exact_or_semantic(javelin_clinical.columns, ["AGE"], [["age"]])
j_sex = exact_or_semantic(javelin_clinical.columns, ["SEX"], [["sex"], ["gender"]])
j_treatment = exact_or_semantic(
    javelin_clinical.columns, ["TRT01P"],
    [["treatment"], ["randomized", "arm"], ["arm"]],
)
j_pdl1 = exact_or_semantic(
    javelin_clinical.columns, ["PDL1FL"],
    [["pdl1"], ["pd", "l1"]],
)
j_cluster = exact_or_semantic(
    javelin_clinical.columns, ["TCGA_cluster"],
    [["tcga", "cluster"], ["molecular", "cluster"], ["cluster"]],
)

javelin_gene_values = javelin_expression.set_index("canonical_id")[JAVELIN_TARGET_GENES]
javelin_gene_values = javelin_gene_values.apply(pd.to_numeric, errors="coerce")
if javelin_gene_values.isna().any().any():
    raise ValueError("JAVELIN target-gene expression contains missing values")
javelin_gene_z = zscore_frame(javelin_gene_values)

javelin_scores = pd.DataFrame(index=javelin_gene_z.index)
javelin_scores["Mitoxy"] = zscore_series(javelin_gene_z[MITOXY_GENES].mean(axis=1))
javelin_scores["Core3"] = zscore_series(javelin_gene_z[CORE3_GENES].mean(axis=1))
javelin_scores["Myeloid trigger"] = zscore_series(
    javelin_gene_z[MYELOID_GENES].mean(axis=1)
)
javelin_scores["Spatial4"] = zscore_series(
    javelin_gene_z[SPATIAL4_GENES].mean(axis=1)
)
javelin_scores["ANGIO"] = zscore_series(javelin_gene_z[ANGIO_GENES].mean(axis=1))
javelin_scores["JAVELIN26"] = zscore_series(
    javelin_gene_z[JAVELIN26_GENES].mean(axis=1)
)
for gene in ["SLC7A11", "BID", "BAK1", "BAX", "RICTOR"]:
    javelin_scores[gene] = javelin_gene_z[gene]
javelin_scores.index.name = "canonical_id"

javelin = javelin_scores.reset_index().merge(
    javelin_clinical, on="canonical_id", how="inner", validate="one_to_one"
)
javelin["pfs"] = pd.to_numeric(javelin[j_pfs], errors="coerce")
javelin["pfs_event"] = parse_event(javelin[j_pfs_status], j_pfs_status)
javelin["age"] = pd.to_numeric(javelin[j_age], errors="coerce")
javelin["sex"] = javelin[j_sex].astype(str).str.strip()
javelin["pdl1"] = javelin[j_pdl1].astype(str).str.strip()
javelin["tcga_cluster"] = javelin[j_cluster].astype(str).str.strip()
javelin["treatment_label"] = javelin[j_treatment].astype(str).str.strip()
treatment_text = javelin["treatment_label"].str.casefold()
javelin["treatment"] = np.select(
    [
        treatment_text.str.contains(r"avelumab|axitinib|comb", regex=True),
        treatment_text.str.contains(r"sunitinib", regex=True),
    ],
    [1.0, 0.0], default=np.nan,
)

javelin_model = javelin.dropna(subset=[
    "pfs", "pfs_event", "age", "sex", "pdl1", "tcga_cluster", "treatment"
]).copy()
if not set(javelin_model["treatment"].unique()).issubset({0.0, 1.0}):
    raise ValueError("JAVELIN treatment coding failed")
if javelin_model["treatment"].nunique() != 2:
    raise ValueError("Both JAVELIN treatment arms were not recovered")

print("JAVELIN matched model rows:", len(javelin_model))
print("JAVELIN PFS events:", int(javelin_model["pfs_event"].sum()))
print("Treatment counts:")
display(javelin_model["treatment_label"].value_counts().to_frame("n"))
display(pd.DataFrame({
    "resolved_field": ["ID", "PFS", "PFS status", "age", "sex", "treatment", "PD-L1", "TCGA cluster"],
    "source_column": [j_id, j_pfs, j_pfs_status, j_age, j_sex, j_treatment, j_pdl1, j_cluster],
}))


JAVELIN matched model rows: 723
JAVELIN PFS events: 356
Treatment counts:


,n
treatment_label,
Sunitinib,371
Avelumab+Axitinib,352


,resolved_field,source_column
0,ID,ID
1,PFS,PFS_P
2,PFS status,PFS_P_CNSR
3,age,AGE
4,sex,SEX
5,treatment,TRT01P
6,PD-L1,PDL1FL
7,TCGA cluster,TCGA_cluster


## Panel A — descriptive two-dimensional JAVELIN subgroups


In [23]:
mitoxy_median = javelin_model["Mitoxy"].median()
immuno_median = javelin_model["JAVELIN26"].median()
javelin_model["mitoxy_group"] = np.where(
    javelin_model["Mitoxy"].ge(mitoxy_median), "high", "low"
)
javelin_model["immuno_group"] = np.where(
    javelin_model["JAVELIN26"].ge(immuno_median), "high", "low"
)

panel_a_rows = []
for mitoxy_group, immuno_group in [
    ("high", "high"), ("high", "low"),
    ("low", "high"), ("low", "low"),
]:
    subset = javelin_model.loc[
        javelin_model["mitoxy_group"].eq(mitoxy_group)
        & javelin_model["immuno_group"].eq(immuno_group)
    ]
    result = cox_treatment_effect(subset)
    panel_a_rows.append({
        "subgroup": (
            f"Mitoxy-{mitoxy_group} / JAVELIN26-{immuno_group}"
        ),
        "n": result["n"], "events": result["events"],
        "hr": result["effect"], "ci_low": result["ci_low"],
        "ci_high": result["ci_high"], "p_value": result["p_value"],
    })

panel_a = pd.DataFrame(panel_a_rows)
validate_intervals(panel_a, "hr")
panel_a.to_csv(OUTPUT_DIR / "panel_a_2d_subgroups.csv", index=False)
display(panel_a)


,subgroup,n,events,hr,ci_low,ci_high,p_value
0,Mitoxy-high / JAVELIN26-high,217,99,0.435622,0.286291,0.662844,0.000104
1,Mitoxy-high / JAVELIN26-low,145,91,0.607068,0.396246,0.930058,0.021840
2,Mitoxy-low / JAVELIN26-high,145,61,0.824319,0.498489,1.363122,0.451537
3,Mitoxy-low / JAVELIN26-low,216,105,0.890176,0.606389,1.306776,0.552543


## Panel B — formal continuous JAVELIN interactions


In [24]:
# The multiplicity family is defined independently of which rows are shown in
# the figure. This prevents the displayed subset from changing its own FDR.
javelin_fdr_family = [
    "Mitoxy", "Core3", "Myeloid trigger", "Spatial4",
    "SLC7A11", "BID", "BAK1", "BAX", "RICTOR",
]

family_rows = []
for score in javelin_fdr_family:
    result = cox_interaction(
        javelin_model, score, "pfs", "pfs_event",
        categorical_covariates=["sex", "pdl1"],
        numeric_covariates=["age"],
    )
    family_rows.append({
        "score": score, "n": result["n"], "events": result["events"],
        "hr": result["effect"], "ci_low": result["ci_low"],
        "ci_high": result["ci_high"], "p_value": result["p_value"],
    })

javelin_family_results = pd.DataFrame(family_rows)
javelin_family_results["fdr"] = bh_adjust(javelin_family_results["p_value"])
javelin_family_results.to_csv(
    OUTPUT_DIR / "panel_b_full_fdr_family.csv", index=False
)

display_order = ["Mitoxy", "Core3", "Myeloid trigger", "SLC7A11"]
panel_b = (
    javelin_family_results.set_index("score").loc[display_order]
    .reset_index()
)
validate_intervals(panel_b, "hr")
panel_b.to_csv(OUTPUT_DIR / "panel_b_formal_interactions.csv", index=False)

print("Full nine-score BH family:")
display(javelin_family_results)
print("Panel B plotting subset:")
display(panel_b)


Full nine-score BH family:


,score,n,events,hr,ci_low,ci_high,p_value,fdr
0,Mitoxy,723,356,0.817003,0.662187,1.008014,0.059357,0.178072
1,Core3,723,356,0.773845,0.623413,0.960577,0.020088,0.178072
2,Myeloid trigger,723,356,0.909293,0.735195,1.124619,0.380534,0.428101
3,Spatial4,723,356,1.101174,0.892590,1.358499,0.368388,0.428101
4,SLC7A11,723,356,0.998049,0.819183,1.215971,0.984538,0.984538
5,BID,723,356,0.837457,0.673224,1.041754,0.111225,0.250257
6,BAK1,723,356,0.886718,0.717681,1.095570,0.265213,0.397819
7,BAX,723,356,0.805506,0.652680,0.994116,0.043907,0.178072
8,RICTOR,723,356,1.139058,0.926593,1.400241,0.216400,0.389520


Panel B plotting subset:


,score,n,events,hr,ci_low,ci_high,p_value,fdr
0,Mitoxy,723,356,0.817003,0.662187,1.008014,0.059357,0.178072
1,Core3,723,356,0.773845,0.623413,0.960577,0.020088,0.178072
2,Myeloid trigger,723,356,0.909293,0.735195,1.124619,0.380534,0.428101
3,SLC7A11,723,356,0.998049,0.819183,1.215971,0.984538,0.984538


## Panel C — exploratory immune/angiogenic context strata


In [25]:
panel_c_rows = []


def stable_context_interaction(data):
    """Fit the prespecified adjusted interaction with a stable Cox start.

    The ridge solution is used only as an optimizer starting point. The
    returned coefficient, confidence interval, and P value all come from the
    final unpenalized partial-likelihood model.
    """
    analysis = data.dropna(subset=["pfs", "pfs_event"]).copy()
    complete, design = prepare_design(
        analysis, "Mitoxy",
        categorical_covariates=["sex", "pdl1", "tcga_cluster"],
        numeric_covariates=["age", "ANGIO", "JAVELIN26"],
    )
    outcomes = analysis.loc[complete.index, ["pfs", "pfs_event"]].astype(float)
    valid = outcomes["pfs"].gt(0) & outcomes["pfs_event"].isin([0, 1])
    outcomes, design = outcomes.loc[valid], design.loc[valid]

    # Retain the three focal terms, then add nuisance columns only when they
    # increase the design rank. This removes empty/sparse dummy columns that
    # can otherwise make the Hessian singular within a median-defined stratum.
    focal = ["score", "treatment", "score_by_treatment"]
    retained = []
    for column in focal + [c for c in design.columns if c not in focal]:
        candidate = retained + [column]
        matrix = design[candidate].to_numpy(dtype=float)
        if np.linalg.matrix_rank(matrix) == len(candidate):
            retained.append(column)
    design = design[retained]
    if "score_by_treatment" not in design.columns:
        raise RuntimeError("The focal interaction is not identifiable")

    model = PHReg(
        outcomes["pfs"], design,
        status=outcomes["pfs_event"], ties="breslow",
    )
    fit = None
    diagnostics = []
    for alpha in (1e-4, 1e-3, 1e-2):
        try:
            ridge = model.fit_regularized(
                alpha=alpha, L1_wt=0.0, maxiter=2000
            )
            candidate = model.fit(
                start_params=np.asarray(ridge.params, dtype=float),
                method="bfgs", maxiter=2000, disp=0,
            )
            interaction_index = design.columns.get_loc("score_by_treatment")
            values = np.array([
                candidate.params[interaction_index],
                candidate.bse[interaction_index],
                candidate.pvalues[interaction_index],
            ], dtype=float)
            if np.isfinite(values).all():
                fit = candidate
                break
            diagnostics.append(f"alpha={alpha}: non-finite inference")
        except Exception as error:
            diagnostics.append(f"alpha={alpha}: {type(error).__name__}: {error}")
    if fit is None:
        raise RuntimeError(
            "Adjusted stratified Cox model did not converge after stable "
            "initialization. " + " | ".join(diagnostics)
        )

    index = design.columns.get_loc("score_by_treatment")
    coefficient = float(fit.params[index])
    standard_error = float(fit.bse[index])
    return {
        "n": len(outcomes),
        "events": int(outcomes["pfs_event"].sum()),
        "effect": float(np.exp(coefficient)),
        "ci_low": float(np.exp(coefficient - 1.96 * standard_error)),
        "ci_high": float(np.exp(coefficient + 1.96 * standard_error)),
        "p_value": float(fit.pvalues[index]),
        "n_parameters": len(retained),
    }


for context, display_context in [
    ("JAVELIN26", "JAVELIN26"), ("ANGIO", "ANGIO")
]:
    median = javelin_model[context].median()
    for stratum, mask in [
        ("Low", javelin_model[context].lt(median)),
        ("High", javelin_model[context].ge(median)),
    ]:
        subset = javelin_model.loc[mask].copy()
        result = stable_context_interaction(subset)
        panel_c_rows.append({
            "context_stratum": f"{display_context}-{stratum.lower()}",
            "context": display_context, "stratum": stratum,
            "n": result["n"], "events": result["events"],
            "hr": result["effect"], "ci_low": result["ci_low"],
            "ci_high": result["ci_high"], "p_value": result["p_value"],
            "n_parameters": result["n_parameters"],
        })

panel_c = pd.DataFrame(panel_c_rows)
validate_intervals(panel_c, "hr")
panel_c.to_csv(OUTPUT_DIR / "panel_c_context_interactions.csv", index=False)
display(panel_c)


,context_stratum,context,stratum,n,events,hr,ci_low,ci_high,p_value,n_parameters
0,JAVELIN26-low,JAVELIN26,Low,361,196,0.924802,0.707583,1.208705,0.567106,12
1,JAVELIN26-high,JAVELIN26,High,362,160,0.642236,0.435594,0.946907,0.025391,12
2,ANGIO-low,ANGIO,Low,361,188,0.803506,0.604333,1.068322,0.132255,12
3,ANGIO-high,ANGIO,High,362,168,0.992065,0.667805,1.473771,0.968528,11


## Reconstruct the CheckMate 025 clinical and expression tables


In [26]:
CHECKMATE_CLINICAL_CACHE = CACHE_DIR / "checkmate_table4_clinical_v2.csv"
CHECKMATE_EXPRESSION_CACHE = CACHE_DIR / "checkmate_table7_target_expression_v2.csv"


def worksheet_rows(worksheet):
    return [list(row) for row in worksheet.iter_rows(values_only=True)]


def reconstruct_checkmate_clinical(path):
    workbook = load_workbook(path, read_only=True, data_only=True)
    sheet_name = resolve_numbered_sheet(workbook.sheetnames, 4)
    rows = worksheet_rows(workbook[sheet_name])
    workbook.close()
    header_scores = []
    for index, row in enumerate(rows[:50]):
        texts = [normalized(value) for value in row if value not in (None, "")]
        score = sum(any(key in text for text in texts) for key in [
            "id", "treatment", "pfs", "os", "response",
        ])
        header_scores.append((score, index))
    score, header_index = max(header_scores)
    if score < 3:
        raise RuntimeError(
            f"Could not identify the CheckMate clinical header in {sheet_name!r}"
        )
    width = max(len(row) for row in rows[header_index:])
    header_row = rows[header_index] + [None] * (width - len(rows[header_index]))
    header = unique_headers(header_row)
    body = [row + [None] * (width - len(row)) for row in rows[header_index + 1:]]
    table = pd.DataFrame(body, columns=header).dropna(how="all").dropna(axis=1, how="all")
    print("CheckMate clinical sheet:", sheet_name)
    print("CheckMate clinical rows:", len(table))
    return table


def reconstruct_checkmate_expression(path, clinical_ids, targets):
    workbook = load_workbook(path, read_only=True, data_only=True)
    sheet_name = resolve_numbered_sheet(workbook.sheetnames, 7)
    worksheet = workbook[sheet_name]
    targets = set(targets)
    first_rows, target_rows = [], {}
    row_count = 0
    for row_index, row in enumerate(worksheet.iter_rows(values_only=True)):
        values = list(row)
        row_count += 1
        if row_index < 30:
            first_rows.append(values)
        gene = next((
            str(value).strip().upper()
            for value in values[:12]
            if str(value).strip().upper() in targets
        ), None)
        if gene is not None:
            target_rows.setdefault(gene, []).append(values)
    workbook.close()

    candidates = []
    for index, row in enumerate(first_rows):
        overlap = sum(canonical_id(value) in clinical_ids for value in row)
        candidates.append((overlap, index, row))
    overlap, header_index, header = max(candidates, key=lambda item: item[0])
    if overlap < 100:
        raise RuntimeError(
            f"CheckMate Table 7 identifier overlap is too small: {overlap}"
        )
    positions = [
        (index, canonical_id(value))
        for index, value in enumerate(header)
        if canonical_id(value) in clinical_ids
    ]
    expression = pd.DataFrame(index=[sample for _, sample in positions])
    for gene, rows in target_rows.items():
        vectors = []
        for row in rows:
            vectors.append([
                pd.to_numeric(row[index], errors="coerce")
                if index < len(row) else np.nan
                for index, _ in positions
            ])
        expression[gene] = np.nanmean(np.asarray(vectors, dtype=float), axis=0)
    expression.index.name = "canonical_id"
    expression = expression.reset_index().drop_duplicates("canonical_id")
    print("CheckMate expression sheet:", sheet_name)
    print("Rows streamed:", row_count)
    print("Detected identifier row:", header_index)
    print("Identifier overlap:", overlap)
    print("Target genes recovered:", sorted(target_rows))
    return expression


if CHECKMATE_CLINICAL_CACHE.exists():
    checkmate_clinical = pd.read_csv(CHECKMATE_CLINICAL_CACHE, dtype=str)
    print("Loading cached CheckMate clinical table.")
else:
    checkmate_clinical = reconstruct_checkmate_clinical(RAW["checkmate"])
    checkmate_clinical.to_csv(CHECKMATE_CLINICAL_CACHE, index=False)

# Table 4 contains more than one identifier-like column. Build the Table 7
# header search from their union, then select the actual join column by direct
# overlap with the recovered expression identifiers.
checkmate_id_candidates = [
    column for column in checkmate_clinical.columns
    if any(token in normalized(column).split() for token in (
        "id", "sample", "patient", "subject"
    ))
]
if not checkmate_id_candidates:
    raise RuntimeError(
        "No identifier-like columns were found in CheckMate Table 4"
    )
checkmate_identifier_universe = set()
for column in checkmate_id_candidates:
    checkmate_identifier_universe.update(
        value for value in checkmate_clinical[column].map(canonical_id)
        if value
    )

if CHECKMATE_EXPRESSION_CACHE.exists():
    checkmate_expression = pd.read_csv(CHECKMATE_EXPRESSION_CACHE)
    cached_genes = set(checkmate_expression.columns) - {"canonical_id"}
    cached_overlap = len(set(
        checkmate_expression["canonical_id"].map(canonical_id)
    ) & checkmate_identifier_universe)
    if not set(CHECKMATE_TARGET_GENES).issubset(cached_genes) or cached_overlap < 100:
        print("Discarding incomplete CheckMate expression cache.")
        CHECKMATE_EXPRESSION_CACHE.unlink()
        checkmate_expression = reconstruct_checkmate_expression(
            RAW["checkmate"], checkmate_identifier_universe,
            CHECKMATE_TARGET_GENES
        )
        checkmate_expression.to_csv(CHECKMATE_EXPRESSION_CACHE, index=False)
    else:
        print("Loading cached CheckMate target-expression table.")
else:
    checkmate_expression = reconstruct_checkmate_expression(
        RAW["checkmate"], checkmate_identifier_universe,
        CHECKMATE_TARGET_GENES
    )
    checkmate_expression.to_csv(CHECKMATE_EXPRESSION_CACHE, index=False)

checkmate_expression["canonical_id"] = checkmate_expression["canonical_id"].map(canonical_id)
expression_ids = set(checkmate_expression["canonical_id"])
identifier_audit = []
for column in checkmate_id_candidates:
    values = checkmate_clinical[column].map(canonical_id)
    identifier_audit.append({
        "column": column,
        "n_nonempty": int(values.str.len().gt(0).sum()),
        "n_unique": int(values.loc[values.str.len().gt(0)].nunique()),
        "expression_overlap": int(len(set(values) & expression_ids)),
    })
identifier_audit = pd.DataFrame(identifier_audit).sort_values(
    ["expression_overlap", "n_unique"], ascending=False
)
display(identifier_audit)
cm_id_column = identifier_audit.iloc[0]["column"]
if int(identifier_audit.iloc[0]["expression_overlap"]) < 100:
    raise RuntimeError(
        "No CheckMate Table 4 identifier column adequately overlaps Table 7"
    )
checkmate_clinical["canonical_id"] = checkmate_clinical[cm_id_column].map(canonical_id)
checkmate_clinical = checkmate_clinical.loc[
    checkmate_clinical["canonical_id"].isin(expression_ids)
].drop_duplicates("canonical_id")
checkmate_ids = set(checkmate_clinical["canonical_id"])
missing = sorted(set(CHECKMATE_TARGET_GENES) - set(checkmate_expression.columns))
if missing:
    raise KeyError(f"CheckMate Table 7 is missing target genes: {missing}")

print("CheckMate clinical identifiers:", len(checkmate_ids))
print("CheckMate expression identifiers:", checkmate_expression["canonical_id"].nunique())
print("CheckMate clinical/expression overlap:", len(
    set(checkmate_expression["canonical_id"]) & checkmate_ids
))


CheckMate clinical sheet: TableS4_cm025_metadata
CheckMate clinical rows: 250
CheckMate expression sheet: TableS7_cm025_TPM
Rows streamed: 30736
Detected identifier row: 1
Identifier overlap: 236
Target genes recovered: ['BAK1', 'BAX', 'BID', 'MAPKAP1', 'MLST8', 'MTOR', 'MYD88', 'RICTOR', 'SLC7A11', 'TLR2', 'TLR4', 'TNF']


,column,n_nonempty,n_unique,expression_overlap
0,RNA_ID,250,250,236


CheckMate clinical identifiers: 236
CheckMate expression identifiers: 236
CheckMate clinical/expression overlap: 236


In [27]:
cm_columns = checkmate_clinical.columns
cm_treatment = select_column(
    cm_columns, [["treatment"], ["treatment", "arm"], ["arm"]]
)
cm_pfs = select_column(
    cm_columns,
    [["pfs", "time"], ["pfs", "month"], ["progression free survival"], ["pfs"]],
    exclude=["cnsr", "censor", "event", "status"],
)
cm_pfs_status = select_column(
    cm_columns,
    [["pfs", "cnsr"], ["pfs", "censor"], ["pfs", "event"], ["pfs", "status"]],
)
cm_os = select_column(
    cm_columns,
    [["os", "time"], ["os", "month"], ["overall survival"], ["os"]],
    exclude=["cnsr", "censor", "event", "status"],
)
cm_os_status = select_column(
    cm_columns,
    [["os", "cnsr"], ["os", "censor"], ["os", "event"], ["os", "status"]],
)
cm_response = select_column(
    cm_columns,
    [["best", "overall", "response"], ["objective", "response"], ["orr"], ["response"]],
    exclude=["duration"],
)

cm_values = checkmate_expression.set_index("canonical_id")[CHECKMATE_TARGET_GENES]
cm_values = cm_values.apply(pd.to_numeric, errors="coerce")
if cm_values.isna().any().any():
    raise ValueError("CheckMate target-gene expression contains missing values")

# Supplementary Table 7 contains non-negative normalized abundance. Apply a
# deterministic log2(x+1) transform only when the released scale is evidently
# unlogged; record the choice in the output for full auditability.
finite_values = cm_values.to_numpy(dtype=float)
if np.nanmin(finite_values) >= 0 and np.nanpercentile(finite_values, 99) > 50:
    cm_model_values = np.log2(cm_values + 1)
    cm_transform = "log2(x + 1)"
else:
    cm_model_values = cm_values.copy()
    cm_transform = "as released"
print("CheckMate expression transform:", cm_transform)

cm_gene_z = zscore_frame(cm_model_values)
cm_scores = pd.DataFrame(index=cm_gene_z.index)
cm_scores["Mitoxy"] = zscore_series(cm_gene_z[MITOXY_GENES].mean(axis=1))
cm_scores["Core3"] = zscore_series(cm_gene_z[CORE3_GENES].mean(axis=1))
cm_scores["Myeloid trigger"] = zscore_series(cm_gene_z[MYELOID_GENES].mean(axis=1))
cm_scores["Spatial4"] = zscore_series(cm_gene_z[SPATIAL4_GENES].mean(axis=1))
for gene in ["BID", "BAK1", "BAX", "SLC7A11", "RICTOR"]:
    cm_scores[gene] = cm_gene_z[gene]
cm_scores.index.name = "canonical_id"

checkmate = cm_scores.reset_index().merge(
    checkmate_clinical, on="canonical_id", how="inner", validate="one_to_one"
)
treatment_text = checkmate[cm_treatment].astype(str).str.strip().str.casefold()
checkmate["treatment"] = np.select(
    [
        treatment_text.str.contains("nivolumab", regex=False),
        treatment_text.str.contains("everolimus", regex=False),
    ],
    [1.0, 0.0], default=np.nan,
)
checkmate["pfs"] = pd.to_numeric(checkmate[cm_pfs], errors="coerce")
checkmate["pfs_event"] = parse_event(checkmate[cm_pfs_status], cm_pfs_status)
checkmate["os"] = pd.to_numeric(checkmate[cm_os], errors="coerce")
checkmate["os_event"] = parse_event(checkmate[cm_os_status], cm_os_status)
checkmate["orr"] = parse_response(checkmate[cm_response])

if checkmate["treatment"].dropna().nunique() != 2:
    raise ValueError(
        "Both CheckMate treatment arms were not recovered. "
        f"Observed labels: {checkmate[cm_treatment].value_counts(dropna=False).to_dict()}"
    )

print("CheckMate matched rows:", len(checkmate))
print("CheckMate treatment counts:")
display(checkmate[cm_treatment].value_counts(dropna=False).to_frame("n"))
display(pd.DataFrame({
    "resolved_field": ["treatment", "PFS", "PFS status", "OS", "OS status", "ORR"],
    "source_column": [cm_treatment, cm_pfs, cm_pfs_status, cm_os, cm_os_status, cm_response],
}))
print("Outcome completeness/events:")
display(pd.DataFrame([
    {"outcome": "ORR", "n": int(checkmate["orr"].notna().sum()),
     "events": int(checkmate["orr"].sum(skipna=True))},
    {"outcome": "PFS", "n": int(checkmate[["pfs", "pfs_event"]].dropna().shape[0]),
     "events": int(checkmate["pfs_event"].sum(skipna=True))},
    {"outcome": "OS", "n": int(checkmate[["os", "os_event"]].dropna().shape[0]),
     "events": int(checkmate["os_event"].sum(skipna=True))},
]))


CheckMate expression transform: as released
CheckMate matched rows: 236
CheckMate treatment counts:


,n
Arm,
EVEROLIMUS,125
NIVOLUMAB,111


,resolved_field,source_column
0,treatment,Arm
1,PFS,PFS
2,PFS status,PFS_CNSR
3,OS,OS
4,OS status,OS_CNSR
5,ORR,ORR


Outcome completeness/events:


,outcome,n,events
0,ORR,206,26
1,PFS,236,27
2,OS,236,55


## Panel D — CheckMate 025 interaction boundary test


In [28]:
checkmate_fdr_family = [
    "Mitoxy", "Core3", "Myeloid trigger", "BID",
    "BAK1", "BAX", "SLC7A11", "RICTOR",
]

cm_family_rows = []
for score in checkmate_fdr_family:
    for outcome in ["ORR", "PFS", "OS"]:
        if outcome == "ORR":
            result = logistic_interaction(checkmate, score, "orr")
        elif outcome == "PFS":
            result = cox_interaction(
                checkmate, score, "pfs", "pfs_event"
            )
        else:
            result = cox_interaction(
                checkmate, score, "os", "os_event"
            )
        cm_family_rows.append({
            "score": score, "outcome": outcome,
            "n": result["n"], "events": result["events"],
            "effect": result["effect"], "ci_low": result["ci_low"],
            "ci_high": result["ci_high"], "p_value": result["p_value"],
        })

checkmate_family_results = pd.DataFrame(cm_family_rows)
checkmate_family_results["fdr"] = bh_adjust(
    checkmate_family_results["p_value"]
)
checkmate_family_results.to_csv(
    OUTPUT_DIR / "panel_d_full_fdr_family.csv", index=False
)

panel_d = (
    checkmate_family_results.loc[
        checkmate_family_results["score"].eq("Mitoxy")
    ]
    .set_index("outcome").loc[["ORR", "PFS", "OS"]]
    .reset_index()
)
validate_intervals(panel_d, "effect")
panel_d.to_csv(OUTPUT_DIR / "panel_d_checkmate025.csv", index=False)

print("Full 24-test BH family:")
display(checkmate_family_results)
print("Panel D plotting subset:")
display(panel_d)


Full 24-test BH family:


,score,outcome,n,events,effect,ci_low,ci_high,p_value,fdr
0,Mitoxy,ORR,206,NaN,2.092985,0.408730,1.071755e+01,0.375438,0.563860
1,Mitoxy,PFS,236,27.0,0.569077,0.239566,1.351818e+00,0.201569,0.522411
2,Mitoxy,OS,236,55.0,0.366602,0.191126,7.031846e-01,0.002531,0.035676
3,Core3,ORR,206,NaN,1.900926,0.426703,8.468478e+00,0.399401,0.563860
4,Core3,PFS,236,27.0,0.533435,0.222592,1.278361e+00,0.158755,0.522411
5,Core3,OS,236,55.0,0.384182,0.204365,7.222162e-01,0.002973,0.035676
6,Myeloid trigger,ORR,206,NaN,2.004086,0.433608,9.262654e+00,0.373412,0.563860
7,Myeloid trigger,PFS,236,27.0,0.815020,0.349647,1.899793e+00,0.635700,0.802989
8,Myeloid trigger,OS,236,55.0,0.708264,0.388977,1.289634e+00,0.259269,0.563860
9,BID,ORR,206,NaN,5.028688,0.514829,4.911862e+01,0.164823,0.522411


Panel D plotting subset:


,outcome,score,n,events,effect,ci_low,ci_high,p_value,fdr
0,ORR,Mitoxy,206,NaN,2.092985,0.408730,10.717550,0.375438,0.563860
1,PFS,Mitoxy,236,27.0,0.569077,0.239566,1.351818,0.201569,0.522411
2,OS,Mitoxy,236,55.0,0.366602,0.191126,0.703185,0.002531,0.035676


## Output audit


In [29]:
required_outputs = {
    "panel_a_2d_subgroups.csv": [
        "subgroup", "n", "events", "hr", "ci_low", "ci_high", "p_value"
    ],
    "panel_b_formal_interactions.csv": [
        "score", "n", "events", "hr", "ci_low", "ci_high", "p_value", "fdr"
    ],
    "panel_c_context_interactions.csv": [
        "context_stratum", "n", "events", "hr", "ci_low", "ci_high", "p_value"
    ],
    "panel_d_checkmate025.csv": [
        "outcome", "n", "effect", "ci_low", "ci_high", "p_value", "fdr"
    ],
}

audit_rows = []
for filename, columns in required_outputs.items():
    path = OUTPUT_DIR / filename
    if not path.is_file():
        raise FileNotFoundError(path)
    table = pd.read_csv(path)
    missing = sorted(set(columns) - set(table.columns))
    if missing:
        raise KeyError(f"{filename} is missing columns: {missing}")
    audit_rows.append({
        "file": filename, "rows": len(table),
        "size_bytes": path.stat().st_size,
        "sha256": sha256_file(path),
    })

output_audit = pd.DataFrame(audit_rows)
output_audit.to_csv(OUTPUT_DIR / "figure_6_output_manifest.csv", index=False)
display(output_audit)
print("\nFigure 6 public-data analysis complete.")
print("Plotting tables saved to:", OUTPUT_DIR)


,file,rows,size_bytes,sha256
0,panel_a_2d_subgroups.csv,4,498,e783e4cbe4ddf55d043e4eb46ee9aa62b79a4275d33e7c...
1,panel_b_formal_interactions.csv,4,500,3fd607685200edd7f4d80f3e20d8076eb17463f673680b...
2,panel_c_context_interactions.csv,4,529,a455cb2a7439048f8e5a05dc23143f1de4c680c66aed2b...
3,panel_d_checkmate025.csv,3,405,4a9f307fe45b1678a52045e14d844f325eedecd435838e...



Figure 6 public-data analysis complete.
Plotting tables saved to: /scratch/ar58064/mitoxy/data/processed/figure_6_clean
